# Ladin Bridge-Transfer — Colab Orchestrator

Runs the full experiment matrix on a T4 GPU.
All results and adapters are checkpointed to Google Drive after each stage.

In [ ]:
# ── 1. Setup environment ─────────────────────────────────────────────────────
import os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/ladin-bridge-transfer'
else:
    DRIVE_ROOT = '/kaggle/working/ladin-bridge-transfer-results'

os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Environment:', 'Kaggle' if IS_KAGGLE else 'Colab')
print('Results root:', DRIVE_ROOT)

In [ ]:
# ── 2. Clone / pull the repo ─────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/omrin23/ladin-bridge-transfer"
REPO_DIR = '/kaggle/working/ladin-bridge-transfer' if IS_KAGGLE else '/content/ladin-bridge-transfer'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

!ls -la

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
import os

# Locate requirements.txt regardless of working directory or cell run order
def _find_req():
    candidates = [
        'requirements.txt',
        '/kaggle/working/ladin-bridge-transfer/requirements.txt',
        '/content/ladin-bridge-transfer/requirements.txt',
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(
        "requirements.txt not found — run cell 2 first to clone the repo"
    )

def _deps_ok():
    try:
        import transformers, peft, datasets, accelerate, sacrebleu
        import torchao
        from packaging.version import Version
        if Version(torchao.__version__) < Version("0.16.0"):
            print(f"torchao {torchao.__version__} is too old (need >=0.16.0) — upgrading…")
            return False
        return True
    except ImportError:
        return False

if _deps_ok():
    print("Dependencies already installed — skipping.")
else:
    import subprocess, sys
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", _find_req()],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("pip install failed — see errors above")
    print("Dependencies installed. Restarting runtime to load them cleanly…")
    print("▶ After the restart, re-run all cells from cell 1.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
# ── 4. HuggingFace login ─────────────────────────────────────────────────────
import huggingface_hub

if IS_COLAB:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
else:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login complete.')

In [ ]:
# ── 5. Verify GPU ────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 6. Configure logging ──────────────────────────────────────────────────────
import sys, os, logging

# Ensure repo root is on the path (guards against cell 2 being skipped after restart)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_DATASETS_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print('HF cache dir:', HF_CACHE)

In [ ]:
# ── 7. Run the full experiment matrix ────────────────────────────────────────
#
# SMOKE RUN: subset_size=500, epochs_ladin=1 (see configs/default.yaml)
# To run for real: set subset_size=18139, epochs_ladin=3 in configs/default.yaml

# Purge any previously imported `src` modules so this cell always runs the
# code currently on disk. Without this, re-running after a `git pull` in
# cell 2 silently keeps executing the OLD code cached in sys.modules.
import sys
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

from src.experiments.run_all import run_experiment

run_experiment(
    drive_root=DRIVE_ROOT,
    run_zero_shot=True,
    run_direct_cond=True,
    run_bridges=True,
    run_similarity=True,
    run_ablation=False,  # SDLad-Ita repo not yet confirmed
    cache_dir=HF_CACHE,
)

In [ ]:
# ── 8. Quick sanity check: print main results table ──────────────────────────
import pandas as pd
from pathlib import Path

results_csv = Path(DRIVE_ROOT) / 'results' / 'main_results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(df.to_string(index=False))
else:
    print('main_results.csv not yet written — run step 7 first.')